# Training Neural Networks: Backpropagation and the Chain Rule

## Introduction

In the previous notebook, we built neural networks from scratch and understood their mathematical structure. But there's a crucial question we haven't answered yet:

**How do neural networks actually learn?**

In this notebook, we'll discover how the **multivariate chain rule** enables networks to learn from examples through a process called **backpropagation**.

## Learning Objectives

By the end of this notebook, you will understand:

1. **Training Data**: What it means to train a network with labeled examples
2. **Cost Functions**: How to measure network performance
3. **The Learning Problem**: Finding optimal weights and biases
4. **Gradient Descent**: Navigating the cost landscape
5. **The Chain Rule in Action**: How backpropagation works
6. **From Simple to Complex**: Extending the method to deep networks

## The Big Idea

Training a neural network is an **optimization problem**: we want to find the weights and biases that minimize the difference between our network's predictions and the true labels. The chain rule gives us the gradients we need to iteratively improve our network!

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

## Part 1: Training Data and Supervised Learning

### What Does "Training" Mean?

When we say we're **training a network**, we mean using **labeled data**: pairs of inputs and their correct outputs.

#### Example: Happy Face Detection

Imagine building a network to recognize happy faces:

| Component | Description | Example |
|-----------|-------------|---------|
| **Input** | Pixel intensities from an image | $[0.2, 0.8, 0.3, \ldots]$ (one value per pixel) |
| **Output** | Labels for face detection and emotion | $[1, 1]$ → "face detected, happy" |
| | | $[1, 0]$ → "face detected, not happy" |
| | | $[0, 0]$ → "no face detected" |

### The Training Process

1. **Initialize**: Start with random weights and biases
2. **Forward Pass**: Feed an input through the network
3. **Measure Error**: Compare output to the correct label
4. **Backward Pass**: Calculate how to adjust weights to reduce error
5. **Update**: Modify weights slightly to improve performance
6. **Repeat**: Do this for many examples until the network learns

### Supervised Learning

This is called **supervised learning** because we provide the "supervision" (correct answers) during training. The network learns by comparing its predictions to these known answers.

## Part 2: The Learning Problem

### A Simple Network Architecture

Let's consider a specific network structure:
- **4 input units** (e.g., 4 features from our data)
- **3 hidden units** (intermediate representation)
- **2 output units** (e.g., binary classification)

### Counting Parameters

How many weights and biases do we need to learn?

#### Weights:
- **Layer 1** (input → hidden): $4 \times 3 = 12$ weights
- **Layer 2** (hidden → output): $3 \times 2 = 6$ weights
- **Total weights**: $12 + 6 = 18$

#### Biases:
- **Hidden layer**: 3 biases
- **Output layer**: 2 biases
- **Total biases**: $3 + 2 = 5$

#### Grand Total: **23 parameters** to learn!

### The Challenge

Our goal is to find the **23 values** (18 weights + 5 biases) that make our network best match the training data. But how?

In [ ]:
# Visualize the network architecture
def visualize_network_architecture():
    """Visualize the 4-3-2 network architecture."""
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Network structure
    layer_sizes = [4, 3, 2]
    layer_names = ['Input\nLayer', 'Hidden\nLayer', 'Output\nLayer']
    
    # Layout parameters
    layer_spacing = 3.0
    neuron_spacing = 1.0
    max_neurons = max(layer_sizes)
    
    neuron_positions = []
    
    # Draw neurons
    for i, (n_neurons, name) in enumerate(zip(layer_sizes, layer_names)):
        x = i * layer_spacing
        y_offset = (max_neurons - n_neurons) * neuron_spacing / 2
        
        layer_pos = []
        for j in range(n_neurons):
            y = j * neuron_spacing + y_offset
            layer_pos.append((x, y))
            
            # Color by layer
            if i == 0:
                color = 'lightgreen'
            elif i == len(layer_sizes) - 1:
                color = 'lightcoral'
            else:
                color = 'lightblue'
            
            circle = plt.Circle((x, y), 0.2, color=color, ec='black', linewidth=2, zorder=2)
            ax.add_patch(circle)
        
        neuron_positions.append(layer_pos)
        
        # Add layer labels
        ax.text(x, -0.9, name, ha='center', va='top', fontsize=12, fontweight='bold')
        ax.text(x, -1.3, f'({n_neurons} neurons)', ha='center', va='top', fontsize=10, style='italic')
    
    # Draw connections (weights)
    for i in range(len(layer_sizes) - 1):
        for x1, y1 in neuron_positions[i]:
            for x2, y2 in neuron_positions[i+1]:
                ax.plot([x1, x2], [y1, y2], 'gray', alpha=0.4, linewidth=0.8, zorder=1)
    
    # Add annotations
    ax.text(1.5, 4.2, '12 weights', ha='center', fontsize=11, 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.text(1.5, 3.9, '3 biases', ha='center', fontsize=10, style='italic')
    
    ax.text(4.5, 4.2, '6 weights', ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.text(4.5, 3.9, '2 biases', ha='center', fontsize=10, style='italic')
    
    # Configure plot
    ax.set_xlim(-0.8, 6.8)
    ax.set_ylim(-2, 4.5)
    ax.axis('off')
    ax.set_aspect('equal')
    
    plt.title('Network Architecture: 4 → 3 → 2\nTotal: 18 weights + 5 biases = 23 parameters', 
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

visualize_network_architecture()

print("Challenge: Find the 23 parameter values that minimize prediction error!")

## Part 3: The Cost Function

### Measuring Network Performance

We need a way to quantify "how wrong" our network is. This is called the **cost function** (also known as loss function or error function).

### Sum of Squared Errors

A common choice is the **sum of squared differences**:

$$C = \sum_{i=1}^{m} (y_i - \hat{y}_i)^2$$

Where:
- $y_i$ is the **desired output** (true label) for training example $i$
- $\hat{y}_i$ is the **actual output** our network produces for example $i$
- $m$ is the number of training examples

### Why Squared Differences?

1. **Always positive**: Errors don't cancel out
2. **Penalizes large errors**: Squared term makes big mistakes much worse
3. **Smooth and differentiable**: Essential for gradient-based optimization
4. **Mathematically convenient**: Leads to nice derivative forms

### The Goal

**Minimize $C$** by adjusting all weights and biases. When $C = 0$, our network perfectly matches the training data!

In [ ]:
# Visualize cost function for a single weight
def simple_cost_function(w, optimal_w=2.5):
    """A simple quadratic cost function with minimum at optimal_w."""
    return (w - optimal_w)**2 + 0.5

# Generate data
w_values = np.linspace(0, 5, 100)
costs = simple_cost_function(w_values)

# Find minimum
optimal_w = 2.5
optimal_cost = simple_cost_function(optimal_w)

# Test points
w0 = 4.0
cost0 = simple_cost_function(w0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Simple convex function
ax1.plot(w_values, costs, 'b-', linewidth=2, label='Cost C(w)')
ax1.plot(optimal_w, optimal_cost, 'g*', markersize=20, label='Global minimum', zorder=5)
ax1.plot(w0, cost0, 'ro', markersize=10, label=f'Current position w={w0}', zorder=5)

# Add gradient arrow
gradient = 2 * (w0 - optimal_w)  # derivative
arrow_length = 0.5
ax1.arrow(w0, cost0, -arrow_length * np.sign(gradient), 0, 
          head_width=0.3, head_length=0.15, fc='red', ec='red', linewidth=2)
ax1.text(w0 - 0.3, cost0 + 0.5, 'Gradient > 0\n→ Decrease w', 
         fontsize=10, ha='center', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

ax1.set_xlabel('Weight (w)', fontsize=12)
ax1.set_ylabel('Cost C(w)', fontsize=12)
ax1.set_title('Ideal Case: Smooth Convex Function', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right plot: Complex wiggly function with local minima
def wiggly_cost(w):
    return 0.3 * (w - 2.5)**2 + 0.5 + 0.8 * np.sin(3*w) * np.exp(-0.1*(w-2.5)**2)

costs_wiggly = [wiggly_cost(w) for w in w_values]

ax2.plot(w_values, costs_wiggly, 'b-', linewidth=2, label='Cost C(w)')

# Find approximate minima
local_mins = [0.8, 2.4, 4.3]
for wmin in local_mins:
    if wmin == 2.4:
        ax2.plot(wmin, wiggly_cost(wmin), 'g*', markersize=20, label='Global minimum', zorder=5)
    else:
        ax2.plot(wmin, wiggly_cost(wmin), 'y*', markersize=15, label='Local minimum' if wmin == 0.8 else '', zorder=5)

ax2.plot(w0, wiggly_cost(w0), 'ro', markersize=10, label=f'Current position w={w0}', zorder=5)

ax2.set_xlabel('Weight (w)', fontsize=12)
ax2.set_ylabel('Cost C(w)', fontsize=12)
ax2.set_title('Reality: Complex Landscape with Local Minima', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Insight: Following the negative gradient leads us downhill,")
print("but in complex landscapes, we might get stuck in local minima!")

## Part 4: Gradient Descent

### The Strategy: Follow the Gradient

If we can compute $\frac{\partial C}{\partial w}$ (the gradient of cost with respect to weight $w$), we know which direction to move:

- If $\frac{\partial C}{\partial w} > 0$: Cost increases as $w$ increases → **decrease** $w$
- If $\frac{\partial C}{\partial w} < 0$: Cost decreases as $w$ increases → **increase** $w$

### The Update Rule

$$w_{\text{new}} = w_{\text{old}} - \alpha \frac{\partial C}{\partial w}$$

Where $\alpha$ is the **learning rate** (how big a step to take).

### The Multi-Dimensional Challenge

But we don't just have one weight—we have **23 parameters**! 

The cost function is actually:

$$C(\mathbf{W}_1, \mathbf{b}_1, \mathbf{W}_2, \mathbf{b}_2)$$

This is a **23-dimensional hypersurface**! We need the **Jacobian** (vector of all partial derivatives):

$$\nabla C = \begin{bmatrix}
\frac{\partial C}{\partial w_1^{(1)}} \\
\frac{\partial C}{\partial w_2^{(1)}} \\
\vdots \\
\frac{\partial C}{\partial b_2^{(2)}}
\end{bmatrix}$$

This gradient vector points in the direction of **steepest ascent**. So we move in the **opposite direction** to go downhill!

In [ ]:
# Visualize 2D gradient descent (weight and bias space)
def cost_2d(w, b):
    """A 2D cost function for visualization."""
    return (w - 2)**2 + (b + 1)**2 + 1

# Create grid
w_range = np.linspace(-1, 5, 100)
b_range = np.linspace(-4, 2, 100)
W_grid, B_grid = np.meshgrid(w_range, b_range)
C_grid = cost_2d(W_grid, B_grid)

# Gradient descent path
def gradient_descent_2d(w0, b0, learning_rate=0.1, n_steps=20):
    """Perform gradient descent and record the path."""
    path_w = [w0]
    path_b = [b0]
    path_c = [cost_2d(w0, b0)]
    
    w, b = w0, b0
    for _ in range(n_steps):
        # Compute gradients (analytical for our simple function)
        grad_w = 2 * (w - 2)
        grad_b = 2 * (b + 1)
        
        # Update
        w = w - learning_rate * grad_w
        b = b - learning_rate * grad_b
        
        path_w.append(w)
        path_b.append(b)
        path_c.append(cost_2d(w, b))
    
    return np.array(path_w), np.array(path_b), np.array(path_c)

# Run gradient descent from a starting point
w0, b0 = 4.5, 1.5
path_w, path_b, path_c = gradient_descent_2d(w0, b0)

# Plotting
fig = plt.figure(figsize=(16, 5))

# Left: Contour plot with gradient descent path
ax1 = fig.add_subplot(131)
contour = ax1.contour(W_grid, B_grid, C_grid, levels=20, cmap='viridis', alpha=0.6)
ax1.clabel(contour, inline=True, fontsize=8)
ax1.plot(path_w, path_b, 'r.-', linewidth=2, markersize=8, label='Gradient descent path')
ax1.plot(path_w[0], path_b[0], 'go', markersize=12, label='Start', zorder=5)
ax1.plot(path_w[-1], path_b[-1], 'b*', markersize=20, label='End', zorder=5)
ax1.plot(2, -1, 'y*', markersize=20, label='True minimum', zorder=5)
ax1.set_xlabel('Weight (w)', fontsize=11)
ax1.set_ylabel('Bias (b)', fontsize=11)
ax1.set_title('Gradient Descent in 2D (w,b) Space', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Middle: 3D surface
ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(W_grid, B_grid, C_grid, cmap='viridis', alpha=0.6, edgecolor='none')
ax2.plot(path_w, path_b, path_c, 'r.-', linewidth=2, markersize=8, label='Descent path')
ax2.scatter(path_w[0], path_b[0], path_c[0], c='green', s=100, label='Start', zorder=5)
ax2.scatter(path_w[-1], path_b[-1], path_c[-1], c='blue', s=200, marker='*', label='End', zorder=5)
ax2.set_xlabel('Weight (w)', fontsize=10)
ax2.set_ylabel('Bias (b)', fontsize=10)
ax2.set_zlabel('Cost C(w,b)', fontsize=10)
ax2.set_title('Cost Surface', fontsize=12, fontweight='bold')
ax2.view_init(elev=25, azim=45)

# Right: Cost over iterations
ax3 = fig.add_subplot(133)
ax3.plot(path_c, 'b.-', linewidth=2, markersize=8)
ax3.set_xlabel('Iteration', fontsize=11)
ax3.set_ylabel('Cost C', fontsize=11)
ax3.set_title('Cost Decreasing Over Time', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Minimum cost')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Started at: w={w0:.2f}, b={b0:.2f}, cost={path_c[0]:.2f}")
print(f"Ended at:   w={path_w[-1]:.2f}, b={path_b[-1]:.2f}, cost={path_c[-1]:.2f}")
print(f"True minimum: w=2.00, b=-1.00, cost=1.00")
print(f"\nCost reduced by {((path_c[0] - path_c[-1])/path_c[0]*100):.1f}%")

## Part 5: The Chain Rule - Backpropagation Unveiled

### The Simple Two-Node Network

Let's start with the simplest possible network to understand backpropagation:

```
a₀ → [w, b] → a₁
```

Where:
- $a_0$ = input
- $a_1 = \sigma(w \cdot a_0 + b)$ = output
- $\sigma$ = activation function (e.g., tanh)
- $C = (y - a_1)^2$ = cost (squared error)

### Computing Gradients with the Chain Rule

We want: $\frac{\partial C}{\partial w}$ and $\frac{\partial C}{\partial b}$

#### For the weight:

$$\frac{\partial C}{\partial w} = \frac{\partial C}{\partial a_1} \cdot \frac{\partial a_1}{\partial w}$$

#### For the bias:

$$\frac{\partial C}{\partial b} = \frac{\partial C}{\partial a_1} \cdot \frac{\partial a_1}{\partial b}$$

Notice: Both share the term $\frac{\partial C}{\partial a_1}$ — this is the gradient flowing back from the cost!

### Introducing the $z$ Variable

For convenience, we introduce an intermediate variable:

$$z_1 = w \cdot a_0 + b$$
$$a_1 = \sigma(z_1)$$

Now our chain becomes **longer**:

$$\frac{\partial C}{\partial w} = \frac{\partial C}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial w}$$

$$\frac{\partial C}{\partial b} = \frac{\partial C}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial b}$$

### Why This Matters

By separating the activation function $\sigma$ from the weighted sum, we can:
1. Compute $\frac{\partial C}{\partial a_1}$ once and reuse it
2. Handle different activation functions easily (just change $\frac{\partial a_1}{\partial z_1}$)
3. Extend to deeper networks systematically

In [ ]:
# Implement backpropagation for a simple 2-node network
class SimpleNeuron:
    """A single neuron with backpropagation."""
    
    def __init__(self, w, b):
        self.w = w
        self.b = b
        # Store intermediate values for backprop
        self.a0 = None
        self.z1 = None
        self.a1 = None
    
    def forward(self, a0):
        """Forward pass: compute output."""
        self.a0 = a0
        self.z1 = self.w * a0 + self.b
        self.a1 = np.tanh(self.z1)
        return self.a1
    
    def backward(self, y):
        """
        Backward pass: compute gradients using chain rule.
        
        Cost: C = (y - a1)^2
        """
        # Step 1: dC/da1
        dC_da1 = -2 * (y - self.a1)
        
        # Step 2: da1/dz1 (derivative of tanh)
        da1_dz1 = 1 - self.a1**2  # derivative of tanh(z) = 1 - tanh²(z)
        
        # Step 3: dz1/dw and dz1/db
        dz1_dw = self.a0
        dz1_db = 1
        
        # Chain rule: combine the pieces
        dC_dw = dC_da1 * da1_dz1 * dz1_dw
        dC_db = dC_da1 * da1_dz1 * dz1_db
        
        return dC_dw, dC_db
    
    def update(self, dC_dw, dC_db, learning_rate):
        """Update weights and biases using gradient descent."""
        self.w -= learning_rate * dC_dw
        self.b -= learning_rate * dC_db

# Create and train a simple neuron
print("=" * 60)
print("BACKPROPAGATION EXAMPLE")
print("=" * 60)
print()

# Initialize neuron
neuron = SimpleNeuron(w=0.5, b=0.2)

# Training example
a0 = 0.8  # input
y = 0.9   # desired output

print(f"Initial parameters: w={neuron.w:.4f}, b={neuron.b:.4f}")
print(f"Training example: input={a0}, desired output={y}")
print()

# Track training
costs = []
weights = []
biases = []

learning_rate = 0.5
n_iterations = 50

for iteration in range(n_iterations):
    # Forward pass
    a1 = neuron.forward(a0)
    cost = (y - a1)**2
    
    # Store for plotting
    costs.append(cost)
    weights.append(neuron.w)
    biases.append(neuron.b)
    
    # Backward pass
    dC_dw, dC_db = neuron.backward(y)
    
    # Update
    neuron.update(dC_dw, dC_db, learning_rate)
    
    if iteration % 10 == 0:
        print(f"Iteration {iteration:2d}: cost={cost:.6f}, w={neuron.w:.4f}, b={neuron.b:.4f}")

print()
print(f"Final parameters: w={neuron.w:.4f}, b={neuron.b:.4f}")
print(f"Final output: {neuron.forward(a0):.4f} (target: {y})")
print(f"Final cost: {costs[-1]:.6f}")
print()

# Visualize training process
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Cost over time
axes[0].plot(costs, 'b-', linewidth=2)
axes[0].set_xlabel('Iteration', fontsize=11)
axes[0].set_ylabel('Cost', fontsize=11)
axes[0].set_title('Cost Decreasing During Training', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Weight trajectory
axes[1].plot(weights, 'g-', linewidth=2)
axes[1].axhline(y=weights[-1], color='red', linestyle='--', alpha=0.5, label='Final value')
axes[1].set_xlabel('Iteration', fontsize=11)
axes[1].set_ylabel('Weight (w)', fontsize=11)
axes[1].set_title('Weight Evolution', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Bias trajectory
axes[2].plot(biases, 'orange', linewidth=2)
axes[2].axhline(y=biases[-1], color='red', linestyle='--', alpha=0.5, label='Final value')
axes[2].set_xlabel('Iteration', fontsize=11)
axes[2].set_ylabel('Bias (b)', fontsize=11)
axes[2].set_title('Bias Evolution', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print("The neuron learned by following the negative gradient!")

## Part 6: Extending to Multi-Layer Networks

### The Power of Backpropagation

The beauty of backpropagation is that it **scales naturally** to deep networks. The chain rule applies layer by layer, working backwards from output to input.

### Two-Layer Network

Consider:
```
a₀ → [W₁, b₁] → a₁ → [W₂, b₂] → a₂ → Cost C
```

To find $\frac{\partial C}{\partial W_1}$, we need to trace back through **two layers**:

$$\frac{\partial C}{\partial W_1} = \frac{\partial C}{\partial a_2} \cdot \frac{\partial a_2}{\partial z_2} \cdot \frac{\partial z_2}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial W_1}$$

### The Backward Pass Algorithm

**Step 1**: Compute gradients at the output layer
- $\frac{\partial C}{\partial a_2}$ ← error signal
- $\frac{\partial a_2}{\partial z_2}$ ← activation derivative

**Step 2**: Propagate backward through hidden layers
- For each layer $l$ (from output to input):
  - Compute $\frac{\partial C}{\partial a_l}$ using gradients from layer $l+1$
  - Compute $\frac{\partial C}{\partial W_l}$ and $\frac{\partial C}{\partial b_l}$

**Step 3**: Update all parameters
- $W_l \leftarrow W_l - \alpha \frac{\partial C}{\partial W_l}$
- $b_l \leftarrow b_l - \alpha \frac{\partial C}{\partial b_l}$

### Why "Backpropagation"?

The name comes from the direction of computation:
1. **Forward pass**: Input → Hidden → Output (compute predictions)
2. **Backward pass**: Output → Hidden → Input (compute gradients)

We **propagate the error backwards** through the network!

### Key Insight

Each layer's gradient depends on the next layer's gradient. By working backwards, we reuse computations efficiently. Without this, computing all gradients would be exponentially expensive!

In [ ]:
# Visualize the computational graph and gradient flow
def visualize_backprop():
    """Visualize forward pass and backward pass."""
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Define positions for computational nodes
    positions = {
        'a0': (0, 0),
        'W1': (1, 0.5),
        'b1': (1, -0.5),
        'z1': (2, 0),
        'σ1': (3, 0),
        'a1': (4, 0),
        'W2': (5, 0.5),
        'b2': (5, -0.5),
        'z2': (6, 0),
        'σ2': (7, 0),
        'a2': (8, 0),
        'y': (9, 0.5),
        'C': (10, 0),
    }
    
    # FORWARD PASS (top plot)
    ax1.set_xlim(-0.5, 10.5)
    ax1.set_ylim(-1.5, 1.5)
    ax1.axis('off')
    ax1.set_title('FORWARD PASS: Computing Predictions', fontsize=14, fontweight='bold', pad=20)
    
    # Draw nodes
    for node, (x, y) in positions.items():
        if node in ['W1', 'b1', 'W2', 'b2', 'y']:
            color = 'lightcoral' if node == 'y' else 'lightyellow'
            shape = 's'  # square for parameters
            size = 600
        elif node in ['z1', 'z2']:
            color = 'lightblue'
            shape = 'o'
            size = 800
        elif node in ['σ1', 'σ2']:
            color = 'lightgreen'
            shape = 'D'  # diamond
            size = 800
        else:
            color = 'lightgray'
            shape = 'o'
            size = 800
        
        ax1.scatter(x, y, c=color, s=size, marker=shape, edgecolors='black', linewidths=2, zorder=3)
        ax1.text(x, y, node, ha='center', va='center', fontsize=11, fontweight='bold', zorder=4)
    
    # Draw forward connections
    forward_edges = [
        ('a0', 'z1'), ('W1', 'z1'), ('b1', 'z1'),
        ('z1', 'σ1'), ('σ1', 'a1'),
        ('a1', 'z2'), ('W2', 'z2'), ('b2', 'z2'),
        ('z2', 'σ2'), ('σ2', 'a2'),
        ('a2', 'C'), ('y', 'C')
    ]
    
    for src, dst in forward_edges:
        x1, y1 = positions[src]
        x2, y2 = positions[dst]
        ax1.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', lw=2, color='blue', alpha=0.6))
    
    # Add layer labels
    ax1.text(0, -1.2, 'Input', ha='center', fontsize=10, style='italic')
    ax1.text(3, -1.2, 'Hidden Layer', ha='center', fontsize=10, style='italic')
    ax1.text(7, -1.2, 'Output Layer', ha='center', fontsize=10, style='italic')
    ax1.text(10, -1.2, 'Cost', ha='center', fontsize=10, style='italic')
    
    # BACKWARD PASS (bottom plot)
    ax2.set_xlim(-0.5, 10.5)
    ax2.set_ylim(-1.5, 1.5)
    ax2.axis('off')
    ax2.set_title('BACKWARD PASS: Computing Gradients (Backpropagation)', fontsize=14, fontweight='bold', pad=20)
    
    # Draw nodes (same as before)
    for node, (x, y) in positions.items():
        if node in ['W1', 'b1', 'W2', 'b2', 'y']:
            color = 'lightcoral' if node == 'y' else 'lightyellow'
            shape = 's'
            size = 600
        elif node in ['z1', 'z2']:
            color = 'lightblue'
            shape = 'o'
            size = 800
        elif node in ['σ1', 'σ2']:
            color = 'lightgreen'
            shape = 'D'
            size = 800
        else:
            color = 'lightgray'
            shape = 'o'
            size = 800
        
        ax2.scatter(x, y, c=color, s=size, marker=shape, edgecolors='black', linewidths=2, zorder=3)
        
        # Add gradient notation
        if node in ['W1', 'b1', 'W2', 'b2']:
            ax2.text(x, y, f'∂C/∂{node}', ha='center', va='center', fontsize=9, fontweight='bold', zorder=4)
        else:
            ax2.text(x, y, node, ha='center', va='center', fontsize=11, fontweight='bold', zorder=4)
    
    # Draw backward connections (reversed)
    backward_edges = [
        ('C', 'a2'), ('C', 'y'),
        ('a2', 'σ2'), ('σ2', 'z2'),
        ('z2', 'W2'), ('z2', 'b2'), ('z2', 'a1'),
        ('a1', 'σ1'), ('σ1', 'z1'),
        ('z1', 'W1'), ('z1', 'b1'), ('z1', 'a0')
    ]
    
    for src, dst in backward_edges:
        x1, y1 = positions[src]
        x2, y2 = positions[dst]
        ax2.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', lw=2, color='red', alpha=0.6))
    
    # Add layer labels
    ax2.text(0, -1.2, 'Input', ha='center', fontsize=10, style='italic')
    ax2.text(3, -1.2, 'Hidden Layer', ha='center', fontsize=10, style='italic')
    ax2.text(7, -1.2, 'Output Layer', ha='center', fontsize=10, style='italic')
    ax2.text(10, -1.2, 'Cost', ha='center', fontsize=10, style='italic')
    
    # Add legend
    ax2.text(0.5, 1.3, '⬤ Activations', fontsize=10, color='gray', transform=ax2.transAxes)
    ax2.text(0.7, 1.3, '⬥ Activation Functions', fontsize=10, color='green', transform=ax2.transAxes)
    ax2.text(0.5, 1.2, '◼ Parameters', fontsize=10, color='goldenrod', transform=ax2.transAxes)
    ax2.text(0.7, 1.2, '◼ Target', fontsize=10, color='lightcoral', transform=ax2.transAxes)
    
    plt.tight_layout()
    plt.show()

visualize_backprop()

print("Forward Pass: Data flows left → right, computing predictions")
print("Backward Pass: Gradients flow right → left, computing how to improve")
print("\nThe chain rule connects everything!")

In [ ]:
# Implement a complete neural network with backpropagation
class NeuralNetwork:
    """A multi-layer neural network with backpropagation."""
    
    def __init__(self, layer_sizes):
        """Initialize network with random weights and biases."""
        self.layer_sizes = layer_sizes
        self.n_layers = len(layer_sizes) - 1
        
        # Initialize parameters
        self.weights = []
        self.biases = []
        
        for i in range(self.n_layers):
            W = np.random.randn(layer_sizes[i+1], layer_sizes[i]) * 0.5
            b = np.random.randn(layer_sizes[i+1]) * 0.1
            self.weights.append(W)
            self.biases.append(b)
        
        # Storage for forward pass (needed for backprop)
        self.z_values = []
        self.activations = []
    
    def sigmoid(self, z):
        """Sigmoid activation function."""
        return np.tanh(z)
    
    def sigmoid_derivative(self, z):
        """Derivative of sigmoid (tanh)."""
        return 1 - np.tanh(z)**2
    
    def forward(self, a):
        """Forward pass through the network."""
        self.activations = [a]
        self.z_values = []
        
        for W, b in zip(self.weights, self.biases):
            z = W @ a + b
            a = self.sigmoid(z)
            
            self.z_values.append(z)
            self.activations.append(a)
        
        return a
    
    def backward(self, y):
        """
        Backward pass: compute gradients using backpropagation.
        
        Returns gradients for all weights and biases.
        """
        m = len(y) if hasattr(y, '__len__') else 1
        
        # Initialize gradient storage
        weight_gradients = [np.zeros_like(W) for W in self.weights]
        bias_gradients = [np.zeros_like(b) for b in self.biases]
        
        # Output layer error: dC/da for the last layer
        # For squared error: C = (y - a)^2, so dC/da = -2(y - a)
        delta = -2 * (y - self.activations[-1])
        
        # Backpropagate through layers (from output to input)
        for l in range(self.n_layers - 1, -1, -1):
            # Gradient of activation function
            delta = delta * self.sigmoid_derivative(self.z_values[l])
            
            # Compute gradients for this layer
            weight_gradients[l] = np.outer(delta, self.activations[l])
            bias_gradients[l] = delta
            
            # Propagate error to previous layer (if not at input)
            if l > 0:
                delta = self.weights[l].T @ delta
        
        return weight_gradients, bias_gradients
    
    def update(self, weight_grads, bias_grads, learning_rate):
        """Update all parameters using gradients."""
        for i in range(self.n_layers):
            self.weights[i] -= learning_rate * weight_grads[i]
            self.biases[i] -= learning_rate * bias_grads[i]
    
    def train_step(self, x, y, learning_rate):
        """Perform one training step: forward, backward, update."""
        # Forward pass
        prediction = self.forward(x)
        
        # Compute cost
        cost = np.sum((y - prediction)**2)
        
        # Backward pass
        weight_grads, bias_grads = self.backward(y)
        
        # Update parameters
        self.update(weight_grads, bias_grads, learning_rate)
        
        return cost

# Example: Train a 4-3-2 network (as mentioned in the transcript)
print("=" * 70)
print("TRAINING A MULTI-LAYER NEURAL NETWORK")
print("=" * 70)
print()

# Create network
nn = NeuralNetwork([4, 3, 2])

print(f"Network architecture: {nn.layer_sizes}")
print(f"Number of parameters: {sum(W.size + b.size for W, b in zip(nn.weights, nn.biases))}")
print()

# Training data (simple example)
X_train = np.array([
    [0.5, -0.3, 0.8, 0.2],
    [0.1, 0.9, -0.2, 0.6],
    [-0.4, 0.3, 0.7, -0.1],
])

y_train = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])

# Train
learning_rate = 0.1
n_epochs = 100
costs_per_epoch = []

print("Training...")
for epoch in range(n_epochs):
    epoch_cost = 0
    for x, y in zip(X_train, y_train):
        cost = nn.train_step(x, y, learning_rate)
        epoch_cost += cost
    
    costs_per_epoch.append(epoch_cost)
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}: Total cost = {epoch_cost:.6f}")

print("\nTraining complete!")
print()

# Test the network
print("Testing trained network:")
for i, (x, y) in enumerate(zip(X_train, y_train)):
    prediction = nn.forward(x)
    print(f"Example {i+1}: Input={x}, Target={y}, Prediction={prediction}, Error={np.linalg.norm(y - prediction):.4f}")

# Visualize training
plt.figure(figsize=(10, 5))
plt.plot(costs_per_epoch, 'b-', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Total Cost', fontsize=12)
plt.title('Cost Decreasing During Training', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nThe network learned by computing gradients via backpropagation!")
print("The chain rule made it all possible!")

## Summary: The Magic of Backpropagation

### What We've Learned

We've seen how neural networks **learn from examples** through an elegant application of calculus:

#### 1. **The Training Problem**
- Start with random weights and biases
- Use labeled training data (input-output pairs)
- Find parameters that minimize prediction error

#### 2. **The Cost Function**
- Measures how wrong the network is: $C = \sum (y - \hat{y})^2$
- Our goal: minimize $C$ by adjusting all parameters

#### 3. **Gradient Descent**
- Follow the negative gradient to go downhill in cost landscape
- Update rule: $w_{\text{new}} = w_{\text{old}} - \alpha \frac{\partial C}{\partial w}$
- Works in 23-dimensional space (for our 4-3-2 network)

#### 4. **The Chain Rule is Key**
- For a simple network: $\frac{\partial C}{\partial w} = \frac{\partial C}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial w}$
- Breaking computation into $z$ (weighted sum) and $a$ (activation) is crucial
- Separates the activation function derivative from the rest

#### 5. **Backpropagation Algorithm**
- **Forward pass**: Compute predictions (left → right)
- **Backward pass**: Compute gradients (right → left)
- Efficiently reuses intermediate computations
- Scales to arbitrarily deep networks

### The Beautiful Connection

Without the **multivariate chain rule**, training neural networks would be computationally infeasible. By working backwards and reusing gradients from later layers, backpropagation makes deep learning possible!

### Key Mathematical Insight

For a multi-layer network, the gradient for layer $l$ depends on gradients from layer $l+1$:

$$\frac{\partial C}{\partial W_l} = \frac{\partial C}{\partial a_{l+1}} \cdot \frac{\partial a_{l+1}}{\partial z_{l+1}} \cdot \frac{\partial z_{l+1}}{\partial a_l} \cdot \frac{\partial a_l}{\partial z_l} \cdot \frac{\partial z_l}{\partial W_l}$$

This recursive structure is what makes backpropagation so powerful and efficient!

### Looking Forward

In practice, we train on **batches** of examples and use more sophisticated optimizers (Adam, RMSprop), but the fundamental idea remains the same: **use the chain rule to compute gradients, then follow them downhill**.

This is calculus in action—and it's what powers modern AI! 🚀

### Historical Note

Backpropagation was popularized in the 1980s (though variants existed earlier). It remains the cornerstone of training neural networks today. The marriage of the chain rule with computational graphs revolutionized machine learning!